In [5]:
import os, glob, re
import numpy as np
import pandas as pd
from lmfit import Parameters, minimize
import warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────────────
BN_STEP_MIN = 100    # don't check for steps before this BN (shared with floor method)
STEP_THRESH = 0.01   # |CE[i]-CE[i-1]| threshold for step artifact (shared with floor method)
TAIL_N      = 50     # empirical floor = mean of last TAIL_N points of fit window (shared)
# ─────────────────────────────────────────────────────────────────────────────────────
# NOTE: There is NO threshold parameter (0.85 / 0.90 / 0.95) here.
# The floor method uses CE_L = CE_o - 0.90*(CE_o - A), which is arbitrary.
# This method replaces it with an automatic elbow found via the perpendicular
# distance method on the actual data — no manual threshold required.
#
# Elbow = the data point with maximum perpendicular distance from the straight
# line connecting start (BN_data[0], CE_data[0]) to end (BN_data[-1], A),
# where A is the empirical asymptote (floor). Both axes are normalized to [0,1]
# before computing distances so BN and CE scales don't interfere.
# ─────────────────────────────────────────────────────────────────────────────────────

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")
print(f"BN_STEP_MIN={BN_STEP_MIN}  STEP_THRESH={STEP_THRESH}  TAIL_N={TAIL_N}")

# Fit-function bounds (same as floor method)
A_MIN, A_MAX = 0.1, 2.3
B_MIN, B_MAX = 0, 1000
N_MIN, N_MAX = 0.5, 2


def initialize_guesses(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    y = y[np.isfinite(y)]
    A0 = np.percentile(y, 5)
    B0 = np.percentile(y, 95) - A0
    n0 = 0.5
    if len(x) > 10:
        denom = y[0] - A0
        if abs(denom) > 1e-10:
            frac = max(1e-6, (y[0] - y[-1]) / denom)
            if frac > 0:
                n0 = max(0.3, min(1.5, -np.log(frac)))
    return A0, n0, B0


def model(params, x):
    vals = params.valuesdict()
    A, B, n = vals["A"], vals["B"], vals["n"]
    return A + B / ((x + 1) ** n)


def residual(params, x, data):
    # A is pinned to the tail floor; weight = x emphasizes the tail (consistent with floor method).
    weight = x
    return weight * (model(params, x) - data)


def empirical_floor(x, y, tail_n=TAIL_N):
    """Empirical asymptote = mean of the last `tail_n` finite points of the fit window."""
    y = np.asarray(y, float)
    y = y[np.isfinite(y)]
    if len(y) == 0:
        return np.nan
    k = int(min(tail_n, len(y)))
    return float(np.mean(y[-k:]))


def fit_curve_floor(x, y, tail_n=TAIL_N):
    """Pin A = empirical tail floor (vary=False); fit only B and n for the curve shape."""
    mask  = ~np.isnan(y)
    x_fit = np.asarray(x)[mask]
    y_fit = np.asarray(y)[mask]
    if len(x_fit) < 10:
        return None, np.nan
    A_floor = empirical_floor(x_fit, y_fit, tail_n=tail_n)
    A0, n0, B0 = initialize_guesses(x_fit, y_fit)
    params = Parameters()
    params.add("A", value=A_floor, vary=False)
    params.add("B", value=max(B0, 1e-6), min=B_MIN, max=B_MAX)
    params.add("n", value=n0, min=N_MIN, max=N_MAX)
    try:
        return minimize(residual, params, args=(x_fit, y_fit)), A_floor
    except Exception:
        return None, A_floor


def fit_quality(x, y, A, B, n, tail_n=TAIL_N):
    """Unweighted RMSE over the whole fit window and over the last `tail_n` points."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 2:
        return np.nan, np.nan, 0
    yhat = A + B / ((x + 1) ** n)
    rmse_full = float(np.sqrt(np.mean((yhat - y) ** 2)))
    k = int(min(tail_n, len(x)))
    rmse_tail = float(np.sqrt(np.mean((yhat[-k:] - y[-k:]) ** 2)))
    return rmse_full, rmse_tail, k


# ── ELBOW DETECTION via perpendicular distance method ────────────────────────────────

def find_elbow_distance(BN_data, CE_data, A):
    """
    Find the elbow of the CE vs BN data using the perpendicular distance method.

    Endpoints of the reference line:
      Start : (BN_data[0],  CE_data[0])   — first data point in the pre-step window
      End   : (BN_data[-1], A)             — last BN in window at the asymptote

    Both axes are normalized to [0, 1]:
      x_norm = (BN - BN[0]) / (BN[-1] - BN[0])
      y_norm = (CE - A)     / (CE[0]  - A)

    In normalized space the reference line runs from (0, 1) to (1, 0),
    with equation  x + y = 1.  The perpendicular distance from a point
    (x_i, y_i) to this line is  |x_i + y_i - 1| / sqrt(2).
    Because the CE curve bows above the line, the signed quantity
    (x_norm + y_norm - 1) is positive for every interior data point;
    maximizing it is equivalent to maximizing the distance.

    Returns (elbow_idx, elbow_BN, elbow_CE).
    Degenerate case (flat curve, e.g. P=100%): returns (None, nan, nan).
    """
    BN_data = np.asarray(BN_data, float)
    CE_data = np.asarray(CE_data, float)

    if len(BN_data) < 2:
        return None, np.nan, np.nan

    BN_range = float(BN_data[-1] - BN_data[0])
    CE_range = float(CE_data[0] - A)    # CE at first point minus asymptote

    if BN_range <= 0 or CE_range <= 1e-10:
        # Degenerate: curve is essentially flat (e.g. P=100%, network never learns)
        return None, np.nan, np.nan

    x_norm = (BN_data - BN_data[0]) / BN_range
    y_norm = (CE_data - A)           / CE_range

    # Signed distance proportional to perpendicular distance from x + y = 1
    distances  = x_norm + y_norm - 1.0
    elbow_idx  = int(np.argmax(distances))
    return elbow_idx, float(BN_data[elbow_idx]), float(CE_data[elbow_idx])


def compute_ipa_elbow(BN_data, CE_data, A):
    """
    Identify the elbow via find_elbow_distance(), then compute IPA.

    With the distance method the elbow IS a data point, so:
      BN_learned = elbow_BN   (no snapping or analytical fallback needed)
      CE_learned = CE_data at the elbow
      IPA        = |CE_o - CE_learned| / BN_learned
    """
    BN_data = np.asarray(BN_data, float)
    CE_data = np.asarray(CE_data, float)

    elbow_idx, elbow_BN, elbow_CE = find_elbow_distance(BN_data, CE_data, A)

    if elbow_idx is None:
        return {"elbow_BN": np.nan, "BN_learned": np.nan,
                "CE_learned": np.nan, "IPA": np.nan}

    BN_learned = elbow_BN
    CE_learned = elbow_CE

    if BN_learned <= 0:
        return {"elbow_BN": elbow_BN, "BN_learned": BN_learned,
                "CE_learned": CE_learned, "IPA": np.nan}

    IPA = abs(CE_o - CE_learned) / BN_learned
    return {"elbow_BN": elbow_BN, "BN_learned": BN_learned,
            "CE_learned": CE_learned, "IPA": IPA}


print("Helpers: find_elbow_distance, compute_ipa_elbow, fit_curve_floor, fit_quality — OK")
print("Cell 1 ready.")

Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
BN_STEP_MIN=100  STEP_THRESH=0.01  TAIL_N=50
Helpers: find_elbow_distance, compute_ipa_elbow, fit_curve_floor, fit_quality — OK
Cell 1 ready.


In [6]:
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 1 (avoid-step, elbow distance) — Batch size {bs}")
    print("=" * 70)
    rows = []
    for p in PRUNING_LEVELS:
        avg_csv = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                               f"averaged_runs_p_{p}_bs_{bs}.csv")
        if not os.path.exists(avg_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — missing {avg_csv}")
            continue
        df = pd.read_csv(avg_csv)
        df.columns = df.columns.str.strip()
        ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
        bn_col = next((c for c in df.columns if "Batch" in c), None)
        if ce_col is None or bn_col is None:
            print(f"  [SKIP] P%={p*100:5.1f}%  — unexpected columns {list(df.columns)}")
            continue
        df = df.dropna(subset=[ce_col, bn_col]).reset_index(drop=True)

        # ── Step detection (identical to floor method) ──────────────────────────────
        bns = df[bn_col].values.astype(float)
        ces = df[ce_col].values.astype(float)
        cutoff_BN     = float(bns[-1])
        step_detected = False
        for i in range(1, len(bns)):
            if bns[i] >= BN_STEP_MIN:
                if abs(ces[i] - ces[i - 1]) > STEP_THRESH:
                    cutoff_BN     = float(bns[i])
                    step_detected = True
                    break
        # ───────────────────────────────────────────────────────────────────────────

        df_fit = df[df[bn_col] < cutoff_BN]
        x = df_fit[bn_col].values.astype(float)
        y = df_fit[ce_col].values.astype(float)

        # ── Fit (identical to floor method: A pinned to empirical tail floor) ──────
        result, A = fit_curve_floor(x, y)
        if result is None:
            print(f"  [FAIL] P%={p*100:5.1f}%  — fit did not converge")
            continue
        B = result.params["B"].value
        n = result.params["n"].value
        rmse_full, rmse_tail, n_tail = fit_quality(x, y, A, B, n)
        # ───────────────────────────────────────────────────────────────────────────

        # ── IPA via elbow distance method (operates on actual data, not fitted curve)
        ipa = compute_ipa_elbow(x, y, A)
        # ───────────────────────────────────────────────────────────────────────────

        step_tag = "[step detected]" if step_detected else "[no step, full data]"
        print(f"  P%={p*100:5.1f}%  cutoff_BN={cutoff_BN:>6.0f} {step_tag:<22}  "
              f"A(floor)={A:.4f}  RMSE_full={rmse_full:7.4f}  "
              f"elbow_BN={ipa['elbow_BN']!r:>8}  BN_learned={ipa['BN_learned']!r:>8}  "
              f"IPA={ipa['IPA']}")

        rows.append({
            "P%":          p * 100,
            "A":           A,
            "B":           B,
            "n":           n,
            "CE_o":        CE_o,
            "elbow_BN":    ipa["elbow_BN"],
            "BN_learned":  ipa["BN_learned"],
            "CE_learned":  ipa["CE_learned"],
            "IPA":         ipa["IPA"],
            "cutoff_BN":   cutoff_BN,
            "RMSE_full":   rmse_full,
            "RMSE_last50": rmse_tail,
            "n_tail":      n_tail,
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_1_fit_params_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")


  Approach 1 (avoid-step, elbow distance) — Batch size 64
  P%=  0.0%  cutoff_BN=   263 [step detected]         A(floor)=0.3397  RMSE_full= 0.3321  elbow_BN=   262.0  BN_learned=   262.0  IPA=0.0074903587094768504
  P%= 10.0%  cutoff_BN=   273 [step detected]         A(floor)=0.3387  RMSE_full= 1.6526  elbow_BN=   272.0  BN_learned=   272.0  IPA=0.007191541833486513
  P%= 20.0%  cutoff_BN=   272 [step detected]         A(floor)=0.3388  RMSE_full= 3.3399  elbow_BN=   271.0  BN_learned=   271.0  IPA=0.007232371191859948
  P%= 30.0%  cutoff_BN=   261 [step detected]         A(floor)=0.3468  RMSE_full= 3.3867  elbow_BN=     0.0  BN_learned=     0.0  IPA=nan
  P%= 40.0%  cutoff_BN=   285 [step detected]         A(floor)=0.3474  RMSE_full= 1.4317  elbow_BN=     0.0  BN_learned=     0.0  IPA=nan
  P%= 50.0%  cutoff_BN=   322 [step detected]         A(floor)=0.3526  RMSE_full= 1.5950  elbow_BN=     0.0  BN_learned=     0.0  IPA=nan
  P%= 60.0%  cutoff_BN=   367 [step detected]         A(floor

In [7]:
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val = np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if not sub.empty else np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_1_elbow_step.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))


Final summary written: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_summary_approach_1_elbow_step.csv
   P%  IPA_Avg_64  STD_64  IPA_Avg_1024  STD_1024  IPA_Avg_60000  STD_60000
  0.0    0.007490     NaN           NaN       NaN       0.012294        NaN
 10.0    0.007192     NaN           NaN       NaN       0.011027        NaN
 20.0    0.007232     NaN           NaN       NaN            NaN        NaN
 30.0         NaN     NaN      0.007075       NaN            NaN        NaN
 40.0         NaN     NaN           NaN       NaN            NaN        NaN
 50.0         NaN     NaN           NaN       NaN            NaN        NaN
 60.0         NaN     NaN           NaN       NaN            NaN        NaN
 70.0         NaN     NaN           NaN       NaN       0.006546        NaN
 80.0         NaN     NaN           NaN       NaN            NaN        NaN
 82.0         NaN     NaN           NaN       NaN            

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TAG   = "1_elbow_step"
TITLE = "Approach 1 (avoid-step, elbow detection) — Fit on averaged raw CE"
BS_COLOR = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

plt.rcParams.update({"font.size": 14})
fig, ax = plt.subplots(figsize=(10, 6))

for bs in BATCH_SIZES:
    mean_col = f"IPA_Avg_{bs}"
    sub = summary_df.dropna(subset=[mean_col])
    if sub.empty:
        continue
    ax.plot(sub["P%"].values, sub[mean_col].values,
            label=f"BS={bs}", color=BS_COLOR[bs], marker="o", markersize=6, linewidth=2)

ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
ax.set_title(TITLE, fontsize=14)
ax.grid(True, which="both", alpha=0.3)
ax.legend(frameon=False)

out_png = os.path.join(OUT_DIR, f"ipa_plot_approach_{TAG}.png")
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_png}")

Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_1\test_1\approach_1_elbow_step\ipa_plot_approach_1_elbow_step.png
